# 😴 Sleep Alarm Detector

A real-time drowsiness detection system built using **OpenCV** and **MediaPipe**. The project monitors eye closure using the Eye Aspect Ratio (EAR) and triggers an alarm when the user's eyes remain closed for too long.

# Imports

This section imports all the required libraries for computer vision, face landmark detection, numerical computation, threading, and alarm functionality.

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
import threading

try:
    import pygame
    PYGAME_AVAILABLE = True
    pygame.mixer.init()
except Exception:
    PYGAME_AVAILABLE = False
    print("[WARNING] pygame could not be initialized.")

pygame 2.6.1 (SDL 2.28.4, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


# Constants

This section defines the threshold values, eye landmark indices, and alarm configuration used throughout the project.

In [2]:
# ==========================
# Project Constants
# ==========================

# Threshold below which the eyes are considered closed.
EAR_THRESHOLD = 0.22

# Time (in seconds) the eyes must remain closed before triggering the alarm.
EYE_CLOSED_SECONDS = 2.5

# Alarm sound file path.
ALARM_SOUND_FILE = "dragon-studio-censor-beep-3-372460.mp3"

# MediaPipe landmark indices for the left and right eyes.
LEFT_EYE_INDICES = [362, 385, 387, 263, 373, 380]
RIGHT_EYE_INDICES = [33, 160, 158, 133, 153, 144]

# Eye Aspect Ratio (EAR)

Calculate the Eye Aspect Ratio (EAR) using facial landmarks to determine whether the eyes are open or closed.

In [3]:
# =====================================
# Calculate Eye Aspect Ratio (EAR)
# =====================================

# Calculate the Eye Aspect Ratio (EAR) using eye landmarks.
def eye_aspect_ratio(landmarks, eye_indices, frame_w, frame_h):
    # Store the coordinates of eye landmarks.
    pts = []

    # Compute the vertical and horizontal eye distances.
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append(np.array([lm.x * frame_w, lm.y * frame_h]))

    A = np.linalg.norm(pts[1] - pts[5])
    B = np.linalg.norm(pts[2] - pts[4])
    C = np.linalg.norm(pts[0] - pts[3])

    ear = (A + B) / (2.0 * C)

    # Return the calculated EAR value.
    return ear

# Alarm Functions

These functions are responsible for playing and stopping the alarm when prolonged eye closure is detected.

In [4]:
# Play the alarm when prolonged eye closure is detected.
def play_alarm():
    if not PYGAME_AVAILABLE:
        print("\a[ALARM] Eyes closed too long! (No sound – pygame not installed)")
        return

    sound_path = os.path.join(os.path.dirname(__file__), ALARM_SOUND_FILE)

    if not os.path.isfile(sound_path):
        print(f"[ALARM] Sound file '{ALARM_SOUND_FILE}' not found – playing beep only.")
        print("\a")
        return

    try:
        if not pygame.mixer.get_init():
            pygame.mixer.init()

        pygame.mixer.music.load(sound_path)
        pygame.mixer.music.play(-1)

    except Exception as e:
        print(f"[ALARM] Could not play sound: {e}")

In [5]:
# Stop the alarm when the user's eyes reopen.
def stop_alarm():

    # Exit if pygame is not available.
    if not PYGAME_AVAILABLE:
        return

    try:
        # Check whether the pygame mixer has been initialized.
        if pygame.mixer.get_init():

            # Stop the alarm only if it is currently playing.
            if pygame.mixer.music.get_busy():
                pygame.mixer.music.stop()

    # Ignore any unexpected pygame errors.
    except Exception:
        pass

# Display Functions

Draw the status overlay, face bounding box, and display real-time information on the video frame.

In [6]:
# Draw the status banner and real-time information on the frame.
def draw_status_overlay(frame, alarm_active, elapsed, ear_avg):
    h, w = frame.shape[:2]
    if alarm_active:
        color = (0, 0, 220)
        bg_color = (0, 0, 180)
        label = "SLEEPING! WAKE UP!"
        icon = "😴"
    else:
        color = (0, 200, 50)
        bg_color = (0, 150, 40)
        label = "AWAKE"
        icon = "👁️"
    if alarm_active:
        pulse = int(abs(np.sin(time.time() * 4)) * 80)
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 255), -1)
        cv2.addWeighted(overlay, 0.20 + pulse / 1000, frame, 0.80 - pulse / 1000, 0, frame)
    pill_w, pill_h = 340, 54
    pill_x = (w - pill_w) // 2
    pill_y = 14
    cv2.rectangle(frame, (pill_x, pill_y), (pill_x + pill_w, pill_y + pill_h), bg_color, -1, cv2.LINE_AA)
    cv2.rectangle(frame, (pill_x, pill_y), (pill_x + pill_w, pill_y + pill_h), color, 2, cv2.LINE_AA)
    cv2.putText(frame, label, (pill_x + 20, pill_y + 36), cv2.FONT_HERSHEY_DUPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    dot_x = pill_x - 30
    dot_y = pill_y + pill_h // 2
    cv2.circle(frame, (dot_x, dot_y), 14, color, -1, cv2.LINE_AA)
    cv2.circle(frame, (dot_x, dot_y), 14, (255, 255, 255), 2, cv2.LINE_AA)
    bar_h = 50
    bar_y = h - bar_h
    cv2.rectangle(frame, (0, bar_y), (w, h), (20, 20, 20), -1)
    ear_text = f"EAR: {ear_avg:.3f}  (thresh: {EAR_THRESHOLD})"
    closed_text = f"Eyes closed: {elapsed:.1f}s / {EYE_CLOSED_SECONDS}s"
    cv2.putText(frame, ear_text, (14, bar_y + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)
    cv2.putText(frame, closed_text, (14, bar_y + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)
    if elapsed > 0 and not alarm_active:
        progress = min(elapsed / EYE_CLOSED_SECONDS, 1.0)
        bar_fill_w = int((w - 28) * progress)
        cv2.rectangle(frame, (14, bar_y + 45), (14 + bar_fill_w, bar_y + 49), (0, 165, 255), -1, cv2.LINE_AA)
    return frame



In [7]:
# Draw a bounding box around the detected face.
def draw_face_box(frame, landmarks, color, w, h):
    xs = [int(lm.x * w) for lm in landmarks]
    ys = [int(lm.y * h) for lm in landmarks]
    pad = 10
    x1, y1 = max(0, min(xs) - pad), max(0, min(ys) - pad)
    x2, y2 = min(w, max(xs) + pad), min(h, max(ys) + pad)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
    corner = 18
    thick = 3
    for cx, cy, dx, dy in [(x1,y1,1,1),(x2,y1,-1,1),(x1,y2,1,-1),(x2,y2,-1,-1)]:
        cv2.line(frame, (cx, cy), (cx + dx*corner, cy), color, thick, cv2.LINE_AA)
        cv2.line(frame, (cx, cy), (cx, cy + dy*corner), color, thick, cv2.LINE_AA)


# Face Detection Initialization

Initialize the MediaPipe Face Mesh model to detect facial landmarks from the webcam stream.

In [8]:
# =====================================
# Initialize MediaPipe Face Mesh
# =====================================

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6,
)

# Main Detection Loop

Capture webcam frames, detect eye closure, calculate the EAR, trigger the alarm when required, and display the processed output.

In [9]:
def main():
    # Initialize the MediaPipe Face Mesh model
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6,
    )

    # Open the default webcam
    cap = cv2.VideoCapture(0)

    # Exit if the webcam cannot be accessed
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return

    # Set webcam resolution
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # Initialize variables for eye closure tracking
    eye_closed_start = None
    alarm_active = False
    alarm_thread = None

    # Record the session start time
    session_start_time = time.time()

    # Display project information
    print("=" * 60)
    print("  Sleep Detector — press Q to quit")
    print(f"  Alarm fires after {EYE_CLOSED_SECONDS}s of closed eyes")
    if PYGAME_AVAILABLE:
        print(f"  Looking for alarm sound: {ALARM_SOUND_FILE}")
    print("=" * 60)

    # Start capturing webcam frames
    while True:
        ret, frame = cap.read()

        # Exit if frame capture fails
        if not ret:
            break

        # Flip the frame for a mirror-like view
        frame = cv2.flip(frame, 1)

        # Get frame dimensions
        h, w = frame.shape[:2]

        # Convert frame from BGR to RGB
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Detect facial landmarks
        results = face_mesh.process(rgb)

        # Default values
        ear_avg = 1.0
        elapsed = 0.0
        face_detected = False

        # Calculate total session duration
        session_time = int(time.time() - session_start_time)
        minutes = session_time // 60
        seconds = session_time % 60

        # Check if a face is detected
        if results.multi_face_landmarks:

            face_detected = True

            # Extract face landmarks
            lms = results.multi_face_landmarks[0].landmark

            # Calculate EAR for both eyes
            ear_l = eye_aspect_ratio(lms, LEFT_EYE_INDICES, w, h)
            ear_r = eye_aspect_ratio(lms, RIGHT_EYE_INDICES, w, h)

            # Compute average EAR
            ear_avg = (ear_l + ear_r) / 2.0

            # Determine whether the eyes are closed
            eyes_closed = ear_avg < EAR_THRESHOLD

            if eyes_closed:

                # Record the time when eyes first close
                if eye_closed_start is None:
                    eye_closed_start = time.time()

                # Calculate how long the eyes have remained closed
                elapsed = time.time() - eye_closed_start

                # Trigger the alarm after the threshold duration
                if elapsed >= EYE_CLOSED_SECONDS and not alarm_active:
                    alarm_active = True
                    alarm_thread = threading.Thread(target=play_alarm, daemon=True)
                    alarm_thread.start()

            else:
                # Reset timer when eyes reopen
                eye_closed_start = None

                # Stop the alarm if it is currently active
                if alarm_active:
                    alarm_active = False
                    stop_alarm()

            # Change face box color based on alarm status
            face_color = (0, 0, 220) if alarm_active else (0, 220, 60)

            # Draw a bounding box around the detected face
            draw_face_box(frame, lms, face_color, w, h)

        else:
            # Reset everything if no face is detected
            eye_closed_start = None

            if alarm_active:
                alarm_active = False
                stop_alarm()

        # Draw the status overlay
        frame = draw_status_overlay(frame, alarm_active, elapsed, ear_avg)

        # Display session timer
        timer_text = f"Session Time: {minutes:02d}:{seconds:02d}"

        (text_w, text_h), _ = cv2.getTextSize(
            timer_text,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            2,
        )

        cv2.putText(
            frame,
            timer_text,
            (w - text_w - 20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2,
            cv2.LINE_AA,
        )

        # Display a message when no face is detected
        if not face_detected:
            cv2.putText(
                frame,
                "No face detected",
                (w // 2 - 130, h // 2),
                cv2.FONT_HERSHEY_DUPLEX,
                1.0,
                (0, 180, 255),
                2,
                cv2.LINE_AA,
            )

        # Display the processed frame
        cv2.imshow("Sleep Alarm Detector", frame)

        # Exit the application when 'Q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release resources and close all windows
    stop_alarm()
    cap.release()
    face_mesh.close()
    cv2.destroyAllWindows()

# Execute the Project

Run the `main()` function to start the real-time sleep detection system. Press **Q** to exit the application.

In [10]:
# =====================================
# Run the project
# =====================================

if __name__ == "__main__":
    main()

  Sleep Detector — press Q to quit
  Alarm fires after 2.5s of closed eyes
  Looking for alarm sound: dragon-studio-censor-beep-3-372460.mp3


C:\Users\lapto\OneDrive\Desktop\PERSONAL\Projects\Sleep-Detect-Alarm\sleep-detect-alarm\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Exception in thread Thread-4 (play_alarm):
Traceback (most recent call last):
  File "C:\Users\lapto\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\lapto\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\lapto\AppData\Local\Temp\ipykernel_1960\3145947783.py", line 7, in play_alarm
NameError: name '__file__' is not defined. Did you mean: '__name__'?


# Conclusion

This project demonstrates how computer vision techniques can be used to build a simple and effective real-time drowsiness detection system using OpenCV and MediaPipe.